# Dự báo lượt thuê xe đạp Seoul bằng Spark

Notebook trình diễn đề tài 4 theo quy trình thu thập, EDA, làm sạch, tạo đặc trưng, mô hình hóa, đánh giá và demo. Dữ liệu thật có 8.760 giờ; các tập 100 nghìn và một triệu hàng chỉ dùng thử tải ETL.

**Cách chạy:** mở notebook từ bản dự án đã tải về, chọn kernel Python 3.12 có các gói trong `requirements-notebook.txt`, rồi chọn **Run All**. Cần Java 17 để chạy Spark. Không cần GPU.

Các ô mặc định huấn luyện lại mô hình và đo đủ 18 lượt benchmark. Kết quả mới được ghi trong thư mục `runs/`, tách khỏi số liệu đã đưa vào báo cáo. Khi tập trình bày, có thể đặt hai cờ ở ô đầu thành `False` để dùng mô hình và benchmark đính kèm; các ô sẽ ghi rõ nguồn kết quả.

Người 1 giới thiệu dữ liệu và mục tiêu; người 2 giải thích làm sạch, đặc trưng; người 3 trình bày mô hình và đánh giá; người 4 chạy phần Spark ETL, benchmark và thử dự báo ở cuối notebook.

Khi mở bản mã nguồn chưa có CSV, ô đầu tải bản UCI chính thức và kiểm tra SHA-256. Bản mã nguồn chạy lần đầu với hai cờ huấn luyện và benchmark đều là True. Gói bàn giao đầy đủ hỗ trợ cả chế độ dùng mô hình đính kèm.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys, os, json, shutil, subprocess, hashlib, time, importlib.util

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Chọn kernel Python 3.12 theo README trước khi chạy notebook.")

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT = next((p for p in candidates if (p / "src/data.py").is_file()
                and (p / "data/DATA_SOURCE.json").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("Mở notebook trong thư mục dự án đã clone hoặc giải nén.")
sys.path.insert(0, str(PROJECT))
if not (PROJECT / "data/raw/SeoulBikeData.csv").is_file():
    subprocess.run([sys.executable, str(PROJECT / "scripts/download_data.py")], cwd=PROJECT, check=True)

os.environ.update(OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2", SPARK_LOCAL_IP="127.0.0.1")
needed = ["numpy", "pandas", "sklearn", "pyspark", "psutil", "matplotlib"]
missing = [name for name in needed if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError("Thiếu " + ", ".join(missing) + ". Cài requirements-notebook.txt rồi khởi động lại kernel.")
java = str(Path(os.environ["JAVA_HOME"]) / "bin/java") if os.environ.get("JAVA_HOME") else shutil.which("java")
if not java:
    raise RuntimeError("Cần cài Java 17 và kiểm tra JAVA_HOME/PATH.")
java_info = subprocess.run([java, "-version"], capture_output=True, text=True, check=True).stderr
if 'version "17.' not in java_info:
    raise RuntimeError("Notebook đã kiểm tra với Java 17. Chọn đúng Java trước khi chạy Spark.")

RETRAIN_MODELS = True
MEASURE_BENCHMARK = True
LAB_SIZES = [8760, 100000, 1000000]
BENCH_SIZES = [8760, 100000, 1000000]
BENCH_REPEATS = 3
if not RETRAIN_MODELS:
    required_models = [PROJECT / "models" / f"{name}.joblib" for name in ["Ridge", "RandomForest", "HistGradientBoosting"]]
    required_models.append(PROJECT / "models/SparkRandomForest/metadata/part-00000")
    if not all(path.is_file() for path in required_models):
        raise RuntimeError("Bản mã nguồn cần chạy lần đầu với RETRAIN_MODELS=True. Chế độ phát lại cần đủ mô hình trong gói bàn giao.")
if not MEASURE_BENCHMARK and not all((PROJECT / "results" / name).is_file() for name in ["benchmark_runs.csv", "benchmark_summary.csv"]):
    raise RuntimeError("Chưa có bảng benchmark đính kèm. Chạy với MEASURE_BENCHMARK=True.")


started = time.perf_counter()
run_id = datetime.now(timezone.utc).strftime("notebook_%Y%m%dT%H%M%S")
RUN = PROJECT / "runs" / run_id
RUN.mkdir(parents=True)
for name in ["src", "models", "tests"]:
    if (PROJECT / name).is_dir():
        shutil.copytree(PROJECT / name, RUN / name, ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
    else:
        (RUN / name).mkdir()
shutil.copy2(PROJECT / "app.py", RUN / "app.py")
for name in ["data/raw", "data/processed", "results", "figures", "logs"]:
    (RUN / name).mkdir(parents=True, exist_ok=True)
shutil.copy2(PROJECT / "data/raw/SeoulBikeData.csv", RUN / "data/raw/SeoulBikeData.csv")
for p in (PROJECT / "results").glob("*"):
    if p.suffix in {".csv", ".json"}:
        shutil.copy2(p, RUN / "results" / p.name)

def run_step(module, *arguments):
    """Chạy module trong tiến trình riêng; lưu log đầy đủ và dừng khi lỗi."""
    result = subprocess.run([sys.executable, "-m", module, *map(str, arguments)],
                            cwd=RUN, env=os.environ.copy(), capture_output=True, text=True)
    text = result.stdout + result.stderr
    (RUN / "logs" / (module.replace(".", "_") + ".log")).write_text(text, encoding="utf-8")
    if result.returncode:
        print(text[-12000:])
        result.check_returncode()
    print(module, "đã chạy xong")
    print(result.stdout[-3500:])

print("Python", sys.version.split()[0])
print(java_info.splitlines()[0])
print("Kết quả mới:", RUN.relative_to(PROJECT))
print("Huấn luyện mới:", RETRAIN_MODELS, "| Đo benchmark mới:", MEASURE_BENCHMARK)


## 1 Thu thập và xác minh nguồn

Nhóm dùng bản **Seoul Bike Sharing Demand** do UCI Machine Learning Repository phân phối, DOI **10.24432/C5F62R**, giấy phép **CC BY 4.0**. Tệp CSV có 14 cột, gồm cả biến mục tiêu `Rented Bike Count`. UCI liệt kê 8.760 quan sát và nhiệm vụ regression.

Ô này đối chiếu SHA-256 với bản nguồn đã ghim. Muốn tải lại từ UCI, chạy `python scripts/download_data.py --verify-only` trong terminal. Bản phân phối, ghi công và thay đổi dẫn xuất nằm trong `data/`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.data import load_data, feature_frame, next_features, validate_hourly
from src.common import FEATURES, FOLDS, TEST_START
from src.metrics import regression_metrics

get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (11, 3.4), "font.family": "DejaVu Sans", "font.size": 10})
source = json.loads((PROJECT / "data/DATA_SOURCE.json").read_text(encoding="utf-8"))
raw_path = PROJECT / "data/raw/SeoulBikeData.csv"
actual_sha = hashlib.sha256(raw_path.read_bytes()).hexdigest()
assert actual_sha == source["sha256"], "CSV không khớp bản UCI đã ghim."
raw = pd.read_csv(raw_path, encoding="latin1", keep_default_na=False)
history = load_data(raw_path)
display(pd.DataFrame([{"Nguồn": source["source"], "DOI": source["doi"],
    "Giấy phép": source["license"], "Số hàng": len(raw), "Số cột": raw.shape[1]}]))
print("SHA-256:", actual_sha)
display(raw.head())


## 2 Khám phá dữ liệu

Khảo sát kiểu cột, khoảng thời gian, phân phối số lượt thuê và chu kỳ theo giờ. `count` là số lượt được ghi nhận trong một giờ, chưa phải nhu cầu riêng của từng trạm hay số người muốn thuê nhưng không có xe. Các biểu đồ toàn năm dùng để mô tả chuỗi; bước huấn luyện ở dưới vẫn giữ thứ tự thời gian.


In [ ]:
display(pd.DataFrame({"Kiểu dữ liệu": raw.dtypes.astype(str),
                      "Ô trống": raw.replace("", np.nan).isna().sum()}))
display(history[["count", "temperature", "humidity", "wind", "rain", "snow"]].describe().round(2))
print("Từ", history.index.min(), "đến", history.index.max())
print("Giờ nhãn 0:", int(history["count"].eq(0).sum()))
display(history.groupby("functioning")["count"].agg(["size", "min", "mean", "max"]).round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
history["count"].hist(bins=40, ax=axes[0], color="#007f85", edgecolor="white")
axes[0].set(xlabel="Lượt thuê trong giờ", ylabel="Số giờ", title="Phân phối biến mục tiêu")
for weekend, label in [(False, "Ngày trong tuần"), (True, "Cuối tuần")]:
    part = history.loc[(history.index.dayofweek >= 5) == weekend]
    part.groupby(part.index.hour)["count"].mean().plot(ax=axes[1], label=label)
axes[1].set(xlabel="Giờ Seoul", ylabel="Lượt thuê trung bình", title="Chu kỳ theo giờ")
axes[1].legend()
plt.tight_layout(); plt.show()


## 3 Làm sạch và kiểm tra lưới giờ

Hai cột `Date`, `Hour` tạo mốc thời gian theo giờ Seoul. Hàm `load_data` kiểm tra đủ 14 cột, kiểu số, giá trị hữu hạn, miền hợp lệ và tính liên tục từng giờ. Tệp sai được báo lỗi để xử lý trước khi tạo lag.

CSV gốc không có ô thiếu hoặc giờ trùng/thiếu. Nhóm giữ 295 giờ nhãn 0 vì chúng gắn với ngừng hoạt động. Do bài này là hồi quy, không đánh giá mất cân bằng lớp hay dùng ma trận nhầm lẫn.


In [ ]:
quality = {
    "Số giờ": len(history),
    "Ô thiếu sau đọc": int(history.isna().sum().sum()),
    "Giờ trùng": int(history.index.duplicated().sum()),
    "Khoảng cách sai một giờ": int(history.index.to_series().diff().iloc[1:].ne(pd.Timedelta(hours=1)).sum()),
    "Giờ nhãn 0": int(history["count"].eq(0).sum()),
    "Giờ nhãn 0 đang hoạt động": int((history["count"].eq(0) & history["functioning"].eq("Yes")).sum()),
}
validate_hourly(history)
display(pd.DataFrame([quality]))
broken = history.drop(history.index[100])
try:
    validate_hourly(broken)
except ValueError as error:
    print("Thử bỏ một giờ — chương trình từ chối:", error)


## 4 Tạo đặc trưng và chia thời gian

Khi dự báo giờ t, nhóm giả định đã chốt số liệu đến t−1. Bộ 21 đầu vào gồm sáu lag, ba rolling mean, năm trường thời tiết trễ một giờ, sáu cột sin/cos cho lịch và cờ cuối tuần. Rolling phải kết thúc ở t−1. 168 giờ đầu chưa đủ lịch sử được bỏ sau bước này.


In [ ]:
x_all = feature_frame(history)
data = x_all.join(history[["count", "functioning"]]).dropna()
assert len(FEATURES) == 21 and len(data) == 8592
display(pd.DataFrame({"Đặc trưng": FEATURES}))
target = pd.Timestamp("2018-10-01 08:00")
example = pd.Series({"lag_1": history.loc[target-pd.Timedelta(hours=1), "count"],
                    "lag_24": history.loc[target-pd.Timedelta(hours=24), "count"],
                    "mean_6": history.loc[target-pd.Timedelta(hours=6):target-pd.Timedelta(hours=1), "count"].mean()})
display(pd.DataFrame({"Tính từ lịch sử": example, "Đặc trưng tạo ra": x_all.loc[target, example.index]}))
print("Ví dụ mean_6 dùng", target-pd.Timedelta(hours=6), "đến", target-pd.Timedelta(hours=1))


In [ ]:
position = 500
altered = history.copy()
altered.loc[history.index[position]:, ["count", "temperature", "humidity", "wind", "rain", "snow"]] = 9999
np.testing.assert_allclose(feature_frame(history).iloc[position], feature_frame(altered).iloc[position])
print("Đổi nhãn/thời tiết từ t trở đi không làm thay đổi X tại t.")

fold_rows = []
for number, (start, end) in enumerate(FOLDS, 1):
    train = data.loc[data.index < start]
    validation = data.loc[(data.index >= start) & (data.index < end)]
    fold_rows.append({"Fold": number, "Train kết thúc": train.index.max(),
                      "Validation bắt đầu": validation.index.min(),
                      "Mẫu train": len(train), "Mẫu validation": len(validation)})
display(pd.DataFrame(fold_rows))
print("Fit cuối:", int((data.index < TEST_START).sum()), "| Test:", int((data.index >= TEST_START).sum()))


## 5 Mô hình hóa

Ba baseline dùng giờ trước, cùng giờ hôm trước và tuần trước. Các ứng viên là Ridge, Random Forest, HistGradientBoosting và Spark Random Forest. Ridge chỉ fit scaler trên tập train của từng fold. Spark ML Pipeline ghép `VectorAssembler` với `RandomForestRegressor`.

Các lệnh dưới chạy đúng mã trong `src/` trên bản dữ liệu làm việc riêng. Với cờ mặc định, chúng huấn luyện lại toàn bộ ứng viên. Giữ nguyên seed và tham số để kiểm tra khả năng tái lập số liệu đã công bố.


In [ ]:
from src.train import candidates as model_candidates
from src.spark_pipeline import estimator
display(pd.DataFrame([{"Mô hình": name, "Cấu hình": str(model)}
                      for name, model in model_candidates().items()]))
print("Spark Pipeline: VectorAssembler -> RandomForestRegressor (60 cây, maxDepth=8, seed=42)")
if RETRAIN_MODELS:
    run_step("src.train")
    run_step("src.spark_pipeline")
else:
    print("ĐANG DÙNG mô hình và kết quả đính kèm; ô này không huấn luyện mới.")


In [ ]:
cv = pd.concat([pd.read_csv(RUN / "results" / f"cv_{engine}.csv")
                for engine in ["sklearn", "spark"]], ignore_index=True)
cv_summary = cv.groupby("model").agg(MAE_cv=("MAE", "mean"),
    Độ_lệch_fold=("MAE", "std"), Số_fold=("fold", "count")).sort_values("MAE_cv")
selected = cv_summary.index[0]
display(cv_summary.round(3))
print("Chọn bằng MAE validation:", selected)
if RETRAIN_MODELS:
    parity = json.loads((RUN / "results/spark_feature_parity.json").read_text())
    print("Spark/pandas:", parity)


## 6 Đánh giá dự báo

Mô hình đã chọn giữ nguyên trọng số trong tháng 10–11. Sau khi mỗi giờ kết thúc, số liệu thật của giờ đó trở thành lịch sử để dự báo giờ tiếp theo. Đây là dự báo một giờ cuốn chiếu, khác với dự báo đồng thời cả 24 giờ.

MAE và RMSE chấm đủ 1.464 giờ. MAPE chia cho nhãn nên chỉ tính trên 1.313 giờ dương, kèm coverage 89,686%. Các giờ bằng 0 vẫn đóng góp vào MAE/RMSE. Precision, Recall, F1 và ROC-AUC không áp dụng cho mục tiêu hồi quy của đề tài này.


In [ ]:
predictions = pd.read_csv(RUN / "results/predictions_sklearn.csv", parse_dates=["timestamp"]).set_index("timestamp")
spark_predictions = pd.read_csv(RUN / "results/predictions_spark.csv", parse_dates=["timestamp"]).set_index("timestamp")
assert predictions.index.equals(spark_predictions.index)
np.testing.assert_array_equal(predictions["count"], spark_predictions["count"])
predictions["SparkRandomForest"] = spark_predictions["prediction"]
model_names = ["Naive_1h", "Naive_24h", "Naive_168h", "Ridge", "RandomForest", "HistGradientBoosting", "SparkRandomForest"]
scores = pd.DataFrame([{"Mô hình": name, **regression_metrics(predictions["count"], predictions[name])}
                       for name in model_names]).set_index("Mô hình")
display(scores[["MAE", "RMSE", "MAPE_positive_pct", "MAPE_n", "MAPE_coverage_pct", "WAPE_pct"]].round(3))
reference_path = PROJECT / "results/test_metrics.csv"
if not reference_path.is_file():
    reference_path = PROJECT / "tests/fixtures/reference_metrics.csv"
published = pd.read_csv(reference_path).set_index("model")
if RETRAIN_MODELS:
    np.testing.assert_allclose(scores.loc[published.index, ["MAE", "RMSE"]],
                               published[["MAE", "RMSE"]], rtol=0, atol=1e-7)
    print("Huấn luyện mới khớp MAE/RMSE của cả bảy mô hình trong báo cáo.")
print("RF có test tốt hơn vẫn được công bố; lựa chọn mặc định giữ theo CV:", selected)


In [ ]:
segment = predictions.iloc[:168][["count", selected, "Naive_1h"]]
ax = segment.rename(columns={"count": "Thực tế", "Naive_1h": "Giờ trước"}).plot(figsize=(12, 3.6))
ax.set(xlabel="Giờ Seoul", ylabel="Lượt thuê", title="Bảy ngày đầu tập kiểm tra")
plt.tight_layout(); plt.show()
display(pd.DataFrame([regression_metrics([0, 10], [5, 8])]).round(3))
print("Ví dụ: MAE=3,5; RMSE≈3,81; MAPE dương=20%; coverage=50%; WAPE=70%.")


## 7 Demo xử lý Big Data và thử dự báo

### Spark ETL và Parquet phân vùng

Mỗi quy mô chạy trong tiến trình riêng: Spark đọc schema, kiểm tra dữ liệu, dùng `range` và broadcast bảng gốc nhỏ để tạo tải, rồi ghi Parquet theo tháng. SQL tổng hợp tháng × giờ nên chỉ đưa tối đa 288 nhóm về Python. Tập lặp có `row_id` và `copy_id` riêng, tách khỏi huấn luyện.


In [ ]:
lab_rows = []
for size in LAB_SIZES:
    run_step("src.bigdata_lab", "--rows", size)
    info = json.loads((RUN / "results/lab" / str(size) / "summary.json").read_text())
    assert info["aggregates_match_pandas"] and info["processed_rows"] == size
    lab_rows.append({"Số hàng": size, "Phân vùng tháng": info["month_partitions"],
                     "Nhóm SQL": info["aggregate_groups"], "Tệp Parquet": info["parquet_files"],
                     "SQL khớp pandas": info["aggregates_match_pandas"], "Có PartitionFilters": info["partition_filter_in_plan"]})
display(pd.DataFrame(lab_rows))


In [ ]:
VIEW_ROWS = LAB_SIZES[0]
MONTH = "2018-10"
if VIEW_ROWS not in LAB_SIZES:
    raise ValueError("VIEW_ROWS phải thuộc LAB_SIZES đã chạy ở ô trên.")
folder = RUN / "results/lab" / str(VIEW_ROWS)
info = json.loads((folder / "summary.json").read_text())
if MONTH not in info["partition_months"]:
    raise ValueError("Chọn MONTH trong: " + ", ".join(info["partition_months"]))

# Đổi MONTH rồi chạy lại ô này sẽ truy vấn Parquet bằng Spark thật.
from src.spark_pipeline import session
from pyspark.sql import functions as F
spark = session("NotebookMonthQuery")
try:
    silver = spark.read.parquet(str(RUN / info["parquet_path"]))
    filtered = silver.filter(F.col("observation_month") == MONTH).select("timestamp", "hour", "count", "temperature")
    query = filtered.groupBy("hour").agg(F.count("*").alias("rows"),
        F.sum("count").alias("total"), F.avg("temperature").alias("avg_temperature")).orderBy("hour")
    monthly = pd.DataFrame([row.asDict() for row in query.collect()]).set_index("hour")
    plan = filtered._jdf.queryExecution().toString()
finally:
    spark.stop()
gold = pd.read_csv(folder / "gold_month_hour.csv")
expected = gold.loc[gold["observation_month"].eq(MONTH)].set_index("hour").sort_index()
np.testing.assert_allclose(monthly[["rows", "total", "avg_temperature"]],
                           expected[["rows", "total", "avg_temperature"]], rtol=1e-10, atol=1e-8)
assert "PartitionFilters" in plan and MONTH in plan

display(monthly.round(3))
monthly["total"].plot.bar(color="#007f85", xlabel="Giờ Seoul", ylabel="Tổng lượt trên tập đang xét",
                          title=f"Spark tổng hợp {MONTH} — {VIEW_ROWS:,} hàng")
plt.tight_layout(); plt.show()
print("Truy vấn Spark thật theo tháng", MONTH, "có", int(monthly["rows"].sum()), "hàng.")
print("Dòng kế hoạch cho thấy lọc phân vùng và cột đọc:")
print("\n".join(line for line in plan.splitlines() if "PartitionFilters" in line or "ReadSchema" in line))
print("Parquet:", info["parquet_path"])
monthly.to_csv(RUN / "results/demo_month.csv")
(RUN / "results/demo_month_plan.txt").write_text(plan, encoding="utf-8")


### So sánh Spark và pandas trên cùng tác vụ

Hai engine đọc cùng CSV, cùng kiểu dữ liệu, lọc và `groupBy` giờ rồi trả 24 nhóm. Có ba quy mô, mỗi engine chạy ba lần trong worker mới, tổng cộng 18 lượt. `etl_s` đo đến khi có kết quả thực; `wall_s` có thêm khởi động và kết thúc. RSS gồm Python và JVM/con, lấy mẫu 25 ms.

Số đo mới phụ thuộc máy, không thay thế số liệu của lần đo 10/11 trong báo cáo. Không xóa OS cache; RSS có thể đếm trùng trang chia sẻ hoặc bỏ lỡ đỉnh ngắn. Kết luận chỉ áp dụng tác vụ và cấu hình `local[2]` đã đo.


In [ ]:
if MEASURE_BENCHMARK:
    run_step("src.benchmark", "--sizes", *BENCH_SIZES, "--repeats", BENCH_REPEATS)
    measurement_source = "Đo mới trên runtime hiện tại"
else:
    measurement_source = "Đọc bảng benchmark đính kèm; không đo mới"
bench = pd.read_csv(RUN / "results/benchmark_summary.csv")
runs = pd.read_csv(RUN / "results/benchmark_runs.csv")
if MEASURE_BENCHMARK:
    assert len(runs) == len(BENCH_SIZES) * 2 * BENCH_REPEATS
    assert runs["aggregates_match"].all() and (runs["peak_rss_mib"] > 0).all()
print(measurement_source)
display(bench.round(3))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, metric, label in [(axes[0], "etl_median_s", "Giây ETL, trung vị"),
                           (axes[1], "peak_rss_median_mib", "MiB RSS đỉnh, trung vị")]:
    bench.pivot(index="size", columns="engine", values=metric).plot.bar(ax=ax, color=["#007f85", "#dd6e42"])
    ax.set(xlabel="Hàng thử tải", ylabel=label)
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()


### Thử dự báo từ lịch sử

Đổi `TARGET` hoặc `MODEL` rồi chạy lại ô dưới. Hàm chỉ nhận lịch sử trước giờ cần dự báo. Mô hình nạp từ thư mục của lần chạy notebook; giá trị thực chỉ dùng để so sánh sau khi dự báo. Ví dụ 03/10/2018 lúc 00:00 cho thấy sai số lớn khi dịch vụ mở lại mà đầu vào chưa có lịch vận hành tương lai.


In [ ]:
import joblib
from src.predict import forecast
TARGET = pd.Timestamp("2018-10-01 08:00")
MODEL = selected
if MODEL == "SparkRandomForest":
    raise ValueError("Ô thử nhanh dùng một trong ba mô hình sklearn; Spark Pipeline đã được chạy và chấm ở phần 5–6.")
model = joblib.load(RUN / "models" / f"{MODEL}.joblib")
timestamp, value, inputs = forecast(history.loc[history.index < TARGET], model=model, name=MODEL)
actual = float(history.loc[TARGET, "count"])
display(pd.DataFrame([{"Mô hình": MODEL, "Giờ dự báo": timestamp, "Lịch sử đến": history.loc[history.index < TARGET].index.max(),
    "Dự báo": value, "Thực tế": actual, "Sai số tuyệt đối": abs(value-actual)}]).round(3))
display(inputs.T.rename(columns={timestamp: "Giá trị"}))
timestamp_next, prediction_next, _ = forecast(history, model=model, name=MODEL)
print("Giờ sau bộ dữ liệu:", timestamp_next, "| Dự báo:", round(prediction_next, 2), "| Chưa có nhãn để chấm.")


### Kiểm tra và lưu kết quả

Ô cuối chạy bộ kiểm thử dữ liệu, chống dùng tương lai, tính chỉ số, nạp model và dashboard phụ trợ. Bản notebook này dùng Linux/Python 3.12/Java 17 trong lần kiểm tra; nhóm vẫn cần tập trên máy trình bày. Hồ sơ gốc từng xem một giai đoạn chồng lấn với test, nên kết quả được gọi là backtest lịch sử.


In [ ]:
run_step("src.analyze")
result = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
                         cwd=RUN, env=os.environ.copy(), capture_output=True, text=True)
(RUN / "logs/tests.log").write_text(result.stdout + result.stderr, encoding="utf-8")
print((result.stdout + result.stderr)[-7000:])
result.check_returncode()
review = {"finished_at_utc": datetime.now(timezone.utc).isoformat(),
          "source_sha256": actual_sha, "retrained_models": RETRAIN_MODELS,
          "measured_benchmark": MEASURE_BENCHMARK, "benchmark_workers": len(runs),
          "lab_sizes": LAB_SIZES, "selected_by_cv": selected,
          "test_hours": len(predictions), "tests_returncode": result.returncode,
          "elapsed_seconds": round(time.perf_counter() - started, 2),
          "scope": "One-host historical backtest. Replicated rows only for ETL load."}
(RUN / "results/notebook_run.json").write_text(json.dumps(review, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame([review]))
print("Đầu ra:", RUN.relative_to(PROJECT))
print("CSV tổng hợp:", (RUN / "results/demo_month.csv").relative_to(PROJECT))


## Tài liệu đối chiếu

- [UCI Seoul Bike Sharing Demand](https://archive.ics.uci.edu/dataset/560/seoul+bike+sharing+demand), DOI 10.24432/C5F62R.
- [Spark 3.5.7 Parquet](https://spark.apache.org/docs/3.5.7/sql-data-sources-parquet.html) và [ML Pipeline](https://spark.apache.org/docs/3.5.7/ml-pipeline.html).
- [scikit-learn lagged features](https://scikit-learn.org/stable/auto_examples/applications/plot_time_series_lagged_features.html).
- Báo cáo bảy phần trong `docs/`; hướng dẫn bốn người trong `docs/ON_TAP_4_NGUOI/`.
